# Relatório Final - Ciência de Dados
## Bacharelado em Ciência da Computação / PUCPR (2026-1)

**Prof. Rayson Laroca**

Alander Menezes Arantes de Ávila - menezes.alander@pucpr.edu.br

Giancarlo Nunes Perli - giancarlo.perli@sanrocco.com.br

Gustavo Faria Cardoso - faria.cardoso@pucpr.edu.br

Paulo Henrique Perin - paulo.perin@pucpr.edu.br

Pedro Lucas Ghezzi Bittencourt - pedro.bittencourt@pucpr.edu.br

## Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew as _skew, kurtosis as _kurt
from matplotlib.patches import Ellipse as MplEllipse
from matplotlib.lines import Line2D

## Carregamento dos Dados

In [ ]:
ds = pd.read_csv(
    "https://github.com/aland3r/asteroids/releases/download/dataset/asteroids.csv",
    low_memory=False
)
print(f"Shape original: {ds.shape}")

## Guia de Atributos

**Identificadores** — `id`, `spkid`, `full_name`, `pdes`, `name`, `prefix`, `orbit_id`: identificam entradas no catálogo, sem valor preditivo.

**Classificação**
- `neo` — *Near-Earth Object*: indica se o asteroide orbita na região próxima à Terra (Y/N).
- `pha` — *Potentially Hazardous Asteroid*: indica se o asteroide é classificado como potencialmente perigoso (Y/N). ⭐ **Variável-alvo do projeto.**

**Características físicas**
- `H` — Magnitude absoluta: medida de brilho intrínseco do asteroide. Quanto menor o valor, mais brilhante e geralmente maior o objeto.
- `diameter` — Diâmetro físico em km.
- `albedo` — Fração da luz solar refletida pela superfície (0 = completamente escuro, 1 = completamente reflexivo).

**Parâmetros orbitais**
- `e` — Excentricidade: descreve o formato da órbita. 0 = circular, próximo de 1 = muito alongada.
- `a` — Semi-eixo maior (AU): distância média do asteroide ao Sol.
- `q` — Periélio (AU): ponto mais próximo do Sol na órbita.
- `ad` — Afélio (AU): ponto mais distante do Sol na órbita.
- `i` — Inclinação (°): ângulo entre o plano da órbita do asteroide e o plano da eclíptica.
- `w` — Argumento do periélio (°): orientação do ponto mais próximo ao Sol dentro do plano orbital.
- `n` — Movimento médio (°/dia): velocidade angular média ao longo da órbita.
- `per_y` — Período orbital em anos.
- `moid` — Distância mínima de interseção orbital (AU): menor distância geométrica possível entre a órbita do asteroide e a da Terra.
- `moid_ld` — Mesmo que `moid`, expresso em distâncias lunares.

**Qualidade do ajuste orbital**
- `sigma_e` — Incerteza formal na excentricidade estimada.
- `rms` — Resíduo do ajuste orbital (arcsec): indica a qualidade do ajuste da órbita calculada às observações reais.

**Outros** — `class`: classe orbital dinâmica (MBA, APO, ATE, AMO, IEO…)

## Objetivos

**Objetivo principal:** Construir um modelo de Machine Learning capaz de classificar, a partir de características físicas e orbitais, se um asteroide é potencialmente perigoso (`pha = Y`).

**Objetivos secundários:**
- Identificar quais variáveis melhor diferenciam PHAs dos demais asteroides.
- Avaliar a viabilidade das variáveis físicas (`diameter`, `albedo`, `H`) como features, considerando a alta taxa de valores ausentes.
- Investigar o desbalanceamento de classes e suas implicações para a modelagem.
- Comparar modelos com e sem `moid_ld`, avaliando se os parâmetros orbitais restantes são suficientes para a classificação.

## Hipóteses

- **H1:** Asteroides com menor distância orbital à Terra (`moid_ld`, `q`) tendem a ser perigosos.
- **H2:** Asteroides maiores (`H` baixo) tendem a ser mais perigosos.
- **H3:** A geometria da órbita (`e`, `a`, `i`) determina se um asteroide cruza a região da Terra.

## Limpeza dos Dados

**Instâncias sem `pha` são úteis para o nosso objetivo?**

Não — `pha` é a variável-alvo. Sem rótulo, a instância não contribui para treino nem avaliação.

In [ ]:
ds_clean = ds.dropna(subset=['pha'])
print(f"Original:              {ds.shape[0]:,} instâncias")
print(f"Após remover nulos em pha: {ds_clean.shape[0]:,} instâncias")

**As colunas de identificação ajudam a prever se um asteroide é perigoso?**

Não — `id`, `spkid`, `full_name`, `pdes`, `name`, `prefix` e `orbit_id` identificam entradas no catálogo mas não descrevem propriedades físicas ou orbitais.

In [ ]:
id_cols = ['id', 'spkid', 'full_name', 'pdes', 'name', 'prefix', 'orbit_id']
ds_clean = ds_clean.drop(columns=id_cols, errors='ignore')
print(f"Colunas após limpeza: {ds_clean.shape[1]}")

**Existem asteroides potencialmente perigosos (`pha=Y`) que não sejam objetos próximos à Terra (`neo=N`)?**

Essa pergunta é importante para decidir se devemos filtrar o dataset para NEOs antes das análises — sem risco de descartar PHAs.

In [ ]:
print("Distribuição de NEO entre os PHAs:")
print(ds_clean[ds_clean['pha'] == 'Y']['neo'].value_counts())

Todos os 2.066 PHAs possuem `neo=Y`. Não existe nenhum PHA com `neo=N` nos dados — as duas classificações são empiricamente inseparáveis.

Para as **análises estatísticas univariadas**, trabalharemos com o dataset completo `ds_clean` — isso preserva o contraste natural entre as populações e mostra as distribuições reais de cada variável.

Para **comparações visuais entre classes** nas análises multivariadas — onde o desbalanceamento distorceria a percepção — usamos `df_sample`: uma amostra igualitária criada abaixo, com todos os PHAs e o mesmo número de não-PHAs sorteados aleatoriamente. Essa amostra é usada **apenas para visualização**.

In [ ]:
# Dataset para análises estatísticas — apenas NEOs
ds_neo = ds_clean[ds_clean['neo'] == 'Y'].copy()

# Separar classes para amostra balanceada
df_y = ds_clean[ds_clean['pha'] == 'Y'].copy()
df_n = ds_clean[ds_clean['pha'] == 'N'].copy()

# Amostra balanceada — usada APENAS em gráficos comparativos entre classes
# onde o desbalanceamento distorceria a percepção visual
n_bal = len(df_y)
df_n_bal = df_n.sample(n=n_bal, random_state=42)
df_sample = pd.concat([df_y, df_n_bal], ignore_index=True)
df_sample = df_sample.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset completo (ds_clean):    {len(ds_clean):,} instâncias")
print(f"Apenas NEOs (ds_neo):            {len(ds_neo):,} instâncias")
print(f"Amostra balanceada (df_sample): {len(df_sample):,} instâncias")
print(f"\nDistribuição PHA no df_sample:")
print(df_sample['pha'].value_counts())

## Descrição Estatística

O dataset contém registros de asteroides catalogados pela NASA. Após a limpeza:

- **938.603 instâncias** no dataset completo
- **38 features** (após remover identificadores)
- **Problema:** classificação binária — `pha = Y` ou `pha = N`
- **Desbalanceamento:** apenas 0,22% dos asteroides são PHAs — qualquer modelo que preveja sempre "não perigoso" acertaria 99,78% das vezes, tornando a acurácia uma métrica inadequada

In [ ]:
print("=== Valores Ausentes (top colunas) ===")
missing = ds_clean.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(ds_clean) * 100).round(2)
miss_df = pd.DataFrame({'ausentes': missing, '%': missing_pct})
print(miss_df[miss_df['ausentes'] > 0].to_string())

print(f"\n=== Tipos de Dados ===")
print(ds_clean.dtypes.value_counts())
print(f"\nColunas numéricas: {ds_clean.select_dtypes(include='number').shape[1]}")
print(f"Colunas categóricas/object: {ds_clean.select_dtypes(include='object').shape[1]}")

In [ ]:
ds_clean.describe().round(4)

In [ ]:
# Cobertura das variáveis físicas
fig, ax = plt.subplots(figsize=(8, 3))

variaveis = ['H', 'diameter', 'albedo']
cobertura = [ds_clean[v].notnull().sum() / len(ds_clean) * 100 for v in variaveis]
cores = ['steelblue', 'salmon', 'salmon']

ax.barh(variaveis, cobertura, color=cores)
for i, c in enumerate(cobertura):
    ax.text(c + 1, i, f'{c:.1f}%', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 115)
ax.set_xlabel('Dados disponíveis (%)')
ax.set_title('Cobertura das variáveis físicas\n(porcentagem de registros com valor preenchido)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()